# 21. Bokeh & Altair — `bokeh · altair`
**Python Libraries Lab · Phase 2 · Data Visualisation**

_Two more interactive styles: building blocks and a grammar of graphics._

✅ **Runs in Colab** as-is.

Bokeh and Altair are two more ways to make interactive charts. Bokeh gives you building blocks, a figure, glyphs like lines and circles, and tools like hover and zoom, so you can build custom interactive views. Altair takes the opposite approach. You describe what the chart means, which column goes on x, which on y, which sets the colour, and Altair works out how to draw it.

**In this notebook:** With Bokeh you will build a line chart with a custom hover tooltip. With Altair you will build a bar chart, then two linked charts where dragging across one filters the other.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q bokeh altair
import importlib.metadata as md
for pkg in ['bokeh', 'altair']:
    print(pkg, md.version(pkg))

## Step 1 · Create the ShopEasy data
Same dataset.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 600
orders = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
    "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n, p=[0.3, 0.2, 0.3, 0.2]),
    "quantity": rng.integers(1, 6, n),
    "price": rng.choice([199, 499, 999, 1499, 2999], n),
    "rating": rng.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
})
orders["revenue"] = orders["quantity"] * orders["price"]
orders.to_csv("shop_orders.csv", index=False)
print(orders.shape, "-> saved shop_orders.csv")
orders.head()

## Step 2 · Bokeh: line chart with hover
`output_notebook()` makes charts appear in Colab. `HoverTool` controls the tooltip.

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
output_notebook()

daily = orders.groupby("date", as_index=False)["revenue"].sum()
source = ColumnDataSource(daily)
p = figure(x_axis_type="datetime", width=750, height=300, title="Daily revenue (hover me)")
p.line("date", "revenue", source=source, line_width=2)
p.scatter("date", "revenue", source=source, size=5)
p.add_tools(HoverTool(tooltips=[("Date", "@date{%d %b}"), ("Revenue", "Rs.@revenue{0,0}")], formatters={"@date": "datetime"}))
show(p)

## Step 3 · Altair: marks and encodings
Read it as: bars, x is category, y is total revenue, colour by category.

In [ ]:
import altair as alt

chart = alt.Chart(orders).mark_bar().encode(
    x="category:N",
    y=alt.Y("sum(revenue):Q", title="Total revenue"),
    color="category:N",
    tooltip=["category", "sum(revenue)"],
).properties(width=400, height=250, title="Revenue by category")
chart

## Step 4 · Altair: linked charts
Drag across the scatter. The bar chart updates to count only the selected orders.

In [ ]:
brush = alt.selection_interval()
points = alt.Chart(orders).mark_circle(opacity=0.6).encode(
    x="price:Q", y="revenue:Q",
    color=alt.condition(brush, "category:N", alt.value("lightgray")),
).add_params(brush).properties(width=380, height=260)
bars = alt.Chart(orders).mark_bar().encode(
    y="category:N", x="count():Q", color="category:N",
).transform_filter(brush).properties(width=380, height=260)
points | bars

## Step 5 · Save both as HTML
Both libraries can export a standalone page.

In [ ]:
from bokeh.io import save
save(p, "bokeh_revenue.html", title="Revenue")
(points | bars).save("altair_linked.html")
print("Saved bokeh_revenue.html and altair_linked.html")

## ✍️ Your turn
Make an Altair **line chart** of total `revenue` per `date`. Store the chart in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    ok = answer is not None and answer.to_dict()["mark"] in ("line", {"type": "line"})
    return ok, "Expected an Altair chart with mark_line()"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = alt.Chart(orders).mark_line().encode(x="date:T", y="sum(revenue):Q")
answer

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **ColumnDataSource**
- **HoverTool**
- **Encoding**
- **Mark**
- **Selection**

**Next:** 22. Streamlit Data Apps (`streamlit`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/22_streamlit.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install bokeh altair
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.